<img src="figures/hyperparams.jpg" alt="Alt text" width="500" height="200">

### Hyperparameters Affecting Model Accuracy

#### Network Topology
- **Number of Nodes**: The number of nodes (neurons) in each layer affects the
  model's ability to learn complex patterns. Too few nodes may lead to
  underfitting, while too many can cause overfitting.

- **Layer Types**: Different types of layers (e.g., dense, convolutional,
  recurrent) are suited for different tasks. Choosing the right type of layer is
  crucial for model performance.

- **Activation Functions**: Activation functions introduce non-linearity into
  the model, helping it learn complex patterns. Common functions include ReLU,
  sigmoid, and tanh.


#### Network Objects
- **Loss Function**: The loss function measures how well the model's predictions
  match the target values. Choosing the right loss function (e.g., MSE for
  regression, CrossEntropy for classification) is essential for training an
  effective model.

- **Optimizer**: The optimizer updates the model's weights based on the loss
  function. Different optimizers (e.g., SGD, Adam) have different strengths and
  can impact the speed and quality of learning.


#### Model Training
- **Learning Rate**: The learning rate controls the size of the steps the
  optimizer takes when updating the weights. A learning rate that's too high can
  cause the model to converge too quickly to a suboptimal solution, while a
  learning rate that's too low can result in slow convergence.

- **Batch Size**: The batch size is the number of samples the model processes
  before updating its weights. Smaller batch sizes can provide more frequent
  updates and potentially better generalization but can be noisy. Larger batch
  sizes provide smoother updates but require more memory and can lead to poorer
  generalization.

- **Number of Epochs**: The number of epochs is the number of times the entire
  training dataset is passed through the model. More epochs can improve
  accuracy, but too many can lead to overfitting.


#### In this notebook, we will be learning rate and epochs as hyperparameters to tune the model.


In [12]:
#%% packages
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader 
import seaborn as sns
from skorch import NeuralNetRegressor
from sklearn.model_selection import GridSearchCV

In [13]:
#%% data import
cars_file = 'https://gist.githubusercontent.com/noamross/e5d3e859aa0c794be10b/raw/b999fb4425b54c63cab088c0ce2c0d6ce961a563/cars.csv'
cars = pd.read_csv(cars_file)

#%% convert data to tensor
X_list = cars.wt.values
X_np = np.array(X_list, dtype=np.float32).reshape(-1,1)
y_list = cars.mpg.values
y_np = np.array(y_list, dtype=np.float32).reshape(-1,1)
X = torch.from_numpy(X_np)
y_true = torch.from_numpy(y_np)

In [14]:
class LinearRegressionTorch(nn.Module):
    def __init__(self, input_size=1, output_size=1):
        super(LinearRegressionTorch, self).__init__()
        self.linear = nn.Linear(input_size, output_size)
    
    def forward(self, x):
        return self.linear(x)
    
input_dim = 1
output_dim = 1
model = LinearRegressionTorch(input_size=input_dim, output_size=output_dim)
model.train()

LinearRegressionTorch(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)

In [15]:
loss_fun = nn.MSELoss()
learning_rate = 0.02
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [16]:
net = NeuralNetRegressor(
    LinearRegressionTorch,
    max_epochs=10,
    lr=0.1,
    # Shuffle training data on each epoch
    iterator_train__shuffle=True,
)
net.set_params(train_split=False, verbose=0)
params = {
    'lr': [0.02, 0.05, 0.08],
    'max_epochs': [10, 200, 500],
}
gs = GridSearchCV(net, params, refit=False, cv=3, scoring='r2', verbose=2)

gs.fit(X, y_true)
print(f"best score: {gs.best_score_:.3f}, best params: {gs.best_params_}")

Fitting 3 folds for each of 9 candidates, totalling 27 fits
[CV] END .............................lr=0.02, max_epochs=10; total time=   0.0s
[CV] END .............................lr=0.02, max_epochs=10; total time=   0.0s
[CV] END .............................lr=0.02, max_epochs=10; total time=   0.0s
[CV] END ............................lr=0.02, max_epochs=200; total time=   0.4s
[CV] END ............................lr=0.02, max_epochs=200; total time=   0.4s
[CV] END ............................lr=0.02, max_epochs=200; total time=   0.4s
[CV] END ............................lr=0.02, max_epochs=500; total time=   1.2s
[CV] END ............................lr=0.02, max_epochs=500; total time=   1.3s
[CV] END ............................lr=0.02, max_epochs=500; total time=   1.1s
[CV] END .............................lr=0.05, max_epochs=10; total time=   0.0s
[CV] END .............................lr=0.05, max_epochs=10; total time=   0.0s
[CV] END .............................lr=0.05, ma

d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\metrics\_regression.py:1220: RuntimeWarning: overflow encountered in square
  numerator = xp.sum(weight * (y_true - y_pred) ** 2, axis=0)
d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\model_selection\_search.py:1102: UserWarning: One or more of the test scores are non-finite: [-2.68192196e+00 -4.75203594e-01  3.72449835e-01 -2.27418303e+00
  3.69068225e-01  5.86964786e-01 -5.70028253e+01 -5.84485487e+19
            -inf]
  warnings.warn(
d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\model_selection\_search.py:1113: RuntimeWarning: invalid value encountered in subtract
  (array - array_means[:, np.newaxis]) ** 2, axis=1, weights=weights
